# 03 — Plugging in your own model

The `deeptumorvqa` framework is designed for one main extension point:
implement a subclass of `HFVLMEvaluator` (for direct VQA) or `AgentEvaluator`
(for tool-using agent), then point the CLI at your module via
`--backend custom --custom-module my_module`.

Below we walk through both.


## A. Custom direct-VQA model

You implement **one method**: `generate(prompt, images, video_path) -> str`.
Everything else (dataset iteration, prompt formatting, scoring, save/resume,
ranking) is handled by the framework.


In [ ]:
# my_vqa_model.py
from deeptumorvqa.eval.vqa_evaluator import HFVLMEvaluator


class MyVQAModel(HFVLMEvaluator):
    def __init__(self):
        super().__init__(model_id="my-org/my-vqa-model")
        # Load your model here
        # self.model = ...
        # self.processor = ...

    def generate(self, prompt, images=None, video_path=None) -> str:
        '''Return the raw model output for one question.

        - `prompt`: assembled user message (already includes patient context,
                    image-modality blurb, MC options or free-form hint)
        - `images`: list of PIL.Image, or None
        - `video_path`: path to .mp4 / .nii.gz file for video / 3D inputs

        Return the raw text — the framework will parse it (extract single
        letter for MC, etc.).
        '''
        # ... your inference code ...
        return "B"  # placeholder


def build_evaluator(args):
    '''Called by the CLI. Receives the parsed args namespace.'''
    return MyVQAModel()


Run it from the CLI:

```bash
deeptumorvqa-eval \
    --backend custom --custom-module my_vqa_model \
    --mode vqa --input 2d_image --format mc \
    --output results/my_vqa.json \
    --label "MyVQAModel"
```


## B. Custom tool-using agent

For agent mode, implement `chat(messages, tools) -> {content, tool_calls}`.
The framework runs the ReAct loop, executes tool calls (looking results up in
the shipped tool cache), and tracks the trajectory.


In [ ]:
# my_agent.py
from deeptumorvqa.eval.agent_evaluator import AgentEvaluator


class MyAgent(AgentEvaluator):
    def __init__(self):
        super().__init__(model_id="my-org/my-agent", mode="oracle")

    def chat(self, messages, tools):
        '''One round-trip with the agent backbone.

        - `messages`: OpenAI-style chat history
        - `tools`:    list of OpenAI tool schemas. EMPTY [] means
                      "this is the last step, give a final answer (no tools)."

        Return:
          {
            "content":    str | None,   # the final answer text (when no tool calls)
            "tool_calls": list[{"name": str, "arguments": dict}],
          }
        '''
        # ... call your model here, e.g. via openai SDK or local generate ...
        # If model emits a tool call -> populate tool_calls
        # If model gave final answer -> content="X", tool_calls=[]
        return {"content": "B", "tool_calls": []}


def build_evaluator(args):
    return MyAgent()


Run it:

```bash
deeptumorvqa-eval \
    --backend custom --custom-module my_agent \
    --mode agent --agent-mode oracle --format mc \
    --output results/my_agent.json
```

For mode `oracle` and `predicted`, the framework injects the cached tool
outputs automatically — your `chat()` only needs to return tool-call
intentions; the framework executes them.

For mode `vision`, the `crop_organ` tool returns a base64-encoded PNG; your
model receives this in the next-turn `tool` message content (re-formatted
to whatever format your model needs).


## What the framework does for you

- Streams the QA records, formats prompts (MC vs free-form, with
  patient/voxel context for measurement subtypes)
- Per-modality lazy download (2D-only eval doesn't pull the 30 GB CT folder)
- Incremental save every 100 questions + `--no-resume` to start fresh
- Tool execution + cache-first lookup (so agent eval needs no segmentation
  model running locally)
- Multi-step ReAct loop with `force_answer_on_last` to prevent infinite loops
- MC + free-form scoring (MRA / binary / categorical, per-subtype-aware)
- Auto leaderboard ranking against the paper's 30 reported models
